# Entregable final — Cashea FPD
## Ronda 1 (parte de validación): sensibilidad de umbrales y bandas

**Idea central:** el modelo estima el riesgo; el negocio decide cuánto riesgo asumir.

Todo lo de esta sección sale de las predicciones **fuera de muestra** de la validación (enero, fold 2 del CV temporal) del modelo Gradient Boosting: `datos/validacion_fold2_gb.csv`. El test no se usa para elegir nada.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path

# Corré el notebook desde la carpeta notebooks/ (o ajustá RAIZ)
RAIZ = Path("..")
val = pd.read_csv(RAIZ / "datos" / "validacion_fold2_gb.csv")
y_val = val["y_val"].to_numpy()
p_val = val["p_val"].to_numpy()
UMBRAL = 0.155   # fijado desde validación
print(len(y_val), "filas | tasa FPD =", round(y_val.mean(), 4))

57426 filas | tasa FPD = 0.0831


### 1. Tabla de sensibilidad (umbral de 0,05 a 0,40, paso 0,005)

Un caso con `prob >= umbral` se va a revisión (positivo predicho).
- `pct_rechazo` = (tp+fp)/N
- `pct_fpd_evitado` = recall = tp/(tp+fn)
- `tasa_fpd_aprobados` = fn/(fn+tn)
- `precision` = tp/(tp+fp)

In [2]:
def fila_umbral(y, p, u):
    pred = p >= u
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    fn = int((~pred & (y == 1)).sum()); tn = int((~pred & (y == 0)).sum())
    n = tp + fp + fn + tn
    return {
        "umbral": u,
        "pct_rechazo": (tp + fp) / n,
        "pct_fpd_evitado": tp / (tp + fn) if (tp + fn) else np.nan,
        "tasa_fpd_aprobados": fn / (fn + tn) if (fn + tn) else np.nan,
        "precision": tp / (tp + fp) if (tp + fp) else np.nan,
        "recall": tp / (tp + fn) if (tp + fn) else np.nan,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }

umbrales = np.round(np.arange(0.05, 0.40 + 1e-9, 0.005), 3)
sens = pd.DataFrame([fila_umbral(y_val, p_val, u) for u in umbrales])

assert len(sens) == 71 and 0.155 in sens["umbral"].values
assert (sens[["tp", "fp", "fn", "tn"]].sum(axis=1) == len(y_val)).all()
assert sens["pct_rechazo"].is_monotonic_decreasing

sens.to_csv(RAIZ / "datos" / "sensibilidad_validacion.csv", index=False)
sens.loc[sens["umbral"] == 0.155]

,umbral,pct_rechazo,pct_fpd_evitado,tasa_fpd_aprobados,precision,recall,tp,fp,fn,tn
21,0.155,0.096141,0.224319,0.071284,0.193805,0.224319,1070,4451,3700,48205


### 2. `corte_medio` y las tres bandas

`corte_medio` es la probabilidad del percentil 70 en validación. Sale de **esta** versión del modelo: si se reentrena, hay que recalcularlo.

In [3]:
corte_medio = float(np.percentile(p_val, 70))
print(f"corte_medio = {corte_medio:.4f}")
assert corte_medio < UMBRAL

banda = np.where(p_val >= UMBRAL, "alto", np.where(p_val >= corte_medio, "medio", "bajo"))
tabla_bandas = (pd.DataFrame({"banda": banda, "y": y_val})
    .groupby("banda")
    .agg(n=("y", "size"), fpd=("y", "sum"), tasa_fpd=("y", "mean"))
    .assign(pct_casos=lambda d: d["n"] / d["n"].sum(),
            pct_del_total_fpd=lambda d: d["fpd"] / d["fpd"].sum())
    .loc[["bajo", "medio", "alto"]])
display(tabla_bandas)
print("FPD que concentra el 30% más riesgoso:", round(y_val[p_val >= corte_medio].sum() / y_val.sum(), 3))

corte_medio = 0.1001


,n,fpd,tasa_fpd,pct_casos,pct_del_total_fpd
banda,,,,,
bajo,40198,2286,0.056869,0.699997,0.479245
medio,11707,1414,0.120782,0.203862,0.296436
alto,5521,1070,0.193805,0.096141,0.224319


FPD que concentra el 30% más riesgoso: 0.521


> **Pendiente (siguiente pasada):** matriz de confusión en test con el umbral 0,155 y `datos/ejemplos.json`. Requieren las solicitudes crudas de test.